# Mapping the Faithfulness of Chain-of-Thought Reasoning

**Research Question**: Can we build a comprehensive taxonomy of CoT failure modes and develop information-theoretic metrics that predict when reasoning steps add genuine signal versus noise?

---

## Pipeline Overview
1. **Setup & Configuration** — Install dependencies, configure paths
2. **Data Download & Parsing** — Download 5 benchmarks, parse into unified format
3. **Preprocessing** — Create splits, format prompts
4. **Baseline Accuracy** — Run all 5 models on all 5 benchmarks
5. **Faithfulness Profiling** — Compute SIG, CNS, RFI metrics
6. **Perturbation Tests** — Early answering, mistake injection, shuffling, deletion, paraphrasing
7. **Failure Taxonomy** — Classify CoT failure modes
8. **Ablation Studies** — Temperature, length, perturbation type, prompt format, model scaling
9. **Cross-Model Analysis** — Inverse scaling hypothesis
10. **Results & Tables** — Generate formatted tables
11. **Visualization** — Heatmaps, radar charts, scaling curves, step-level plots
12. **Sampling** — Sample and inspect individual CoT examples

## 1. Setup & Configuration

In [1]:
# Install dependencies (run once)
!pip install -r requirements.txt
#!pip install torch torchaudio torchvision numpy datasets pandas matplotlib tqdm typing seaborn dataclasses dotenv transformers

In [2]:
!pip install packaging ninja

  Using cached ninja-1.13.0-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (5.1 kB)


In [3]:
!pip install flash-attn --no-build-isolation

  Using cached flash_attn-2.8.3.tar.gz (8.4 MB)
  Preparing metadata (pyproject.toml) ... error
  error: subprocess-exited-with-error
  
  × Preparing metadata (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [33 lines of output]
      /home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/wheel/bdist_wheel.py:4: FutureWarning: The 'wheel' package is no longer the canonical location of the 'bdist_wheel' command, and will be removed in a future release. Please update to setuptools v70.1 or later which contains an integrated version of this command.
        warn(
      
      
      torch.__version__  = 2.8.0+cu128
      
      
      <string>:106: UserWarning: flash_attn was requested, but nvcc was not found.  Are you sure your environment has nvcc available?  If you're installing within a container from https://hub.docker.com/r/pytorch/pytorch, only images whose names contain 'devel' will provide nvcc.
      Traceback (most recent call last):
        File "

In [1]:
import os
import sys
import json
import random
import numpy as np
import torch
import warnings
warnings.filterwarnings('ignore')

# Ensure project root is in path
PROJECT_ROOT = os.path.dirname(os.path.abspath('__file__'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# Import configs
from configs.model_config import MODEL_REGISTRY, GENERATION_CONFIG, get_model_config
from configs.benchmark_config import BENCHMARK_REGISTRY, get_benchmark_config
from configs.experiment_config import EXPERIMENT_CONFIG, PATHS, ensure_dirs

# Create all directories
ensure_dirs()

# Set seeds for reproducibility
SEED = EXPERIMENT_CONFIG['seed']
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True

print(f'Project Root: {PROJECT_ROOT}')
print(f'Device: {EXPERIMENT_CONFIG["device"]}')
print(f'Precision: {EXPERIMENT_CONFIG["dtype"]}')
print(f'AMP: {EXPERIMENT_CONFIG["use_amp"]}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')
print(f'\nModels: {list(MODEL_REGISTRY.keys())}')
print(f'Benchmarks: {list(BENCHMARK_REGISTRY.keys())}')

Project Root: /teamspace/studios/this_studio
Device: cuda
Precision: bfloat16
AMP: True

Models: ['ds-r1-qwen-7b', 'ds-r1-llama-8b', 'ds-r1-qwen-14b', 'qwq-32b', 'ds-r1-qwen-32b']
Benchmarks: ['gsm8k', 'math', 'strategyqa', 'arc_challenge', 'folio']


In [2]:
import sys
sys.path.insert(0, "./")

In [3]:
os.getcwd()

'/teamspace/studios/this_studio'

In [ ]:
from huggingface_hub import login

login("")

## 2. Data Download & Parsing

In [5]:
from src.data.download_datasets import download_all_datasets

# Download all 5 benchmark datasets
download_info = download_all_datasets(
    output_dir=PATHS['raw_data_dir'],
    cache_dir=PATHS['cache_dir'],
)

print("\n=== Download Summary ===")

for bench, info in download_info.items():
    print(f"\n{bench}:")

    if "error" in info:
        print(f"  ❌ {info['error']}")
        continue

    splits = info.get("splits", [])
    sizes = info.get("sizes", {})

    # SAFE handling
    if isinstance(splits, list):
        for s in splits:
            print(f"  {bench}/{s}: {sizes.get(s, '?')} examples")
    else:
        print(f"  {bench}: invalid splits format -> {type(splits)}")


  ✅ Done: ['train', 'test']
    train: 7473
    test: 1319

  ✅ Done: ['train']
    train: 12500

  ✅ Done: ['train', 'test']
    train: 1603
    test: 687

  ✅ Done: ['train', 'test', 'validation']
    train: 1119
    test: 1172
    validation: 299

  ✅ Done: ['train', 'validation']
    train: 1001
    validation: 203

=== Summary ===
gsm8k: ['train', 'test']
math: ['train']
strategyqa: ['train', 'test']
arc-challenge: ['train', 'test', 'validation']
folio: ['train', 'validation']

=== Download Summary ===

gsm8k:
  gsm8k/train: 7473 examples
  gsm8k/test: 1319 examples

math:
  math/train: 12500 examples

strategyqa:
  strategyqa/train: 1603 examples
  strategyqa/test: 687 examples

arc-challenge:
  arc-challenge/train: 1119 examples
  arc-challenge/test: 1172 examples
  arc-challenge/validation: 299 examples

folio:
  folio/train: 1001 examples
  folio/validation: 203 examples


In [6]:
from datasets import load_dataset
import re
from src.data.dataset_loader import DatasetLoader
from src.data.preprocessing import preprocess_dataset
from configs.benchmark_config import BENCHMARK_REGISTRY

processed_data = {}

for bench_key, config in BENCHMARK_REGISTRY.items():
    print(f"\nLoading {bench_key}...")

    loader = DatasetLoader(config)
    data = loader.load()

    data = data.select(range(min(200, len(data))))  # subsample

    processed = preprocess_dataset(data, bench_key, "zero_shot_cot")

    processed_data[bench_key] = processed

print("\nprocessed_data ready:")
for k, v in processed_data.items():
    print(f"{k}: {len(v)} examples")


Loading gsm8k...


Loaded openai/gsm8k [test] → 1319 samples

Loading math...
Loaded qwedsacf/competition_math [train] → 12500 samples

Loading strategyqa...
Loaded ChilleD/StrategyQA [test] → 687 samples

Loading arc_challenge...
Loaded allenai/ai2_arc [test] → 1172 samples

Loading folio...
Loaded tasksource/folio [validation] → 203 samples

processed_data ready:
gsm8k: 200 examples
math: 200 examples
strategyqa: 200 examples
arc_challenge: 200 examples
folio: 200 examples


In [ ]:
from concurrent.futures import ThreadPoolExecutor
from src.models.model_loader import ModelManager
from src.models.inference import InferenceEngine
from src.utils.answer_extractor import AnswerExtractor
from src.utils.cache import get_cache, save_cache, _hash

def run_model(model_key):
    model_config = MODEL_REGISTRY[model_key]

    print(f'\n{"="*60}')
    print(f'Model: {model_config.short_name} ({model_config.params_b}B)')
    print(f'{"="*60}')

    model_manager = ModelManager(cache_dir=PATHS.get('model_cache_dir'))
    answer_extractor = AnswerExtractor()

    model, tokenizer = model_manager.load_model(model_config)
    gen_config = model_manager.get_generation_config()

    engine = InferenceEngine(
        model, tokenizer, gen_config,
        device=EXPERIMENT_CONFIG['device'],
        use_amp=EXPERIMENT_CONFIG['use_amp'],
    )

    model_results = {}

    for bench_key, examples in processed_data.items():
        correct = 0
        predictions = []

        batch_size = model_config.batch_size
        prompts = [ex['prompt'] for ex in examples]

        cache_hits = {}
        new_prompts = []
        new_indices = []

        # Check cache
        for i, prompt in enumerate(prompts):
            key = _hash(prompt)
            cached = get_cache(key)

            if cached:
                cache_hits[i] = cached
            else:
                new_prompts.append(prompt)
                new_indices.append(i)

        # Run only uncached
        if new_prompts:
            new_outputs = engine.generate_batch(
                new_prompts,
                batch_size=batch_size
            )

            for idx, out in zip(new_indices, new_outputs):
                key = _hash(prompts[idx])
                save_cache(key, out)
                cache_hits[idx] = out

        # Reconstruct full outputs
        outputs = [cache_hits[i] for i in range(len(prompts))]

        for i, (ex, output) in enumerate(zip(examples, outputs)):
            predicted = engine.extract_answer(
                output['raw_output'], ex['answer_type'], bench_key
            )

            is_correct = answer_extractor.check_answer(
                predicted, ex['gold_answer'], ex['answer_type']
            )

            if is_correct:
                correct += 1

            predictions.append({
                'id': ex.get('id', f'{bench_key}_{i}'),
                'gold_answer': ex['gold_answer'],
                'predicted_answer': predicted,
                'is_correct': is_correct,
                'raw_output': output['raw_output'][:500],
                'num_steps': output['num_steps'],
                'num_tokens': output['num_generated_tokens'],
            })

        accuracy = correct / max(1, len(examples)) * 100
        model_results[bench_key] = {
            'accuracy': accuracy,
            'correct': correct,
            'total': len(examples),
            'predictions': predictions,
        }

        print(f'  {bench_key}: {accuracy:.2f}%')

    # Save
    os.makedirs(os.path.join(PATHS['raw_results_dir'], 'baseline'), exist_ok=True)

    save_path = os.path.join(PATHS['raw_results_dir'], 'baseline', f'baseline_{model_key}.json')
    summary = {k: {kk: vv for kk, vv in v.items() if kk != 'predictions'} for k, v in model_results.items()}

    with open(save_path, 'w') as f:
        json.dump({'model': model_config.short_name, 'results': summary}, f, indent=2)

    pred_path = os.path.join(PATHS['raw_results_dir'], 'baseline', f'predictions_{model_key}.json')
    with open(pred_path, 'w') as f:
        json.dump({'model': model_config.short_name, 'predictions': {
            k: v['predictions'] for k, v in model_results.items()
        }}, f, indent=2)

    model_manager.unload_model()
    return model_results

In [ ]:
baseline_results = {}

# ⚠️ IMPORTANT
MAX_WORKERS = 2   # DO NOT increase yet

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    results = list(executor.map(run_model, MODEL_REGISTRY.keys()))

# map back
for model_key, res in zip(MODEL_REGISTRY.keys(), results):
    baseline_results[model_key] = res

print('\n=== Baseline Complete ===')


Model: DS-R1-Qwen-7B (7.0B)
[2026-04-15 10:04:05] [INFO] [model_loader] Loading model DS-R1-Qwen-7B (deepseek-ai/DeepSeek-R1-Distill-Qwen-7B) in float16...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

[2026-04-15 10:04:08] [INFO] [model_loader] GPU Memory: 14.2GB allocated, 14.2GB reserved
[2026-04-15 10:04:08] [INFO] [model_loader] Successfully loaded DS-R1-Qwen-7B


The following generation flags are not valid and may be ignored: ['temperature', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


...

Fetching 14 files:   0%|          | 0/14 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/771 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Some parameters are on the meta device because they were offloaded to the cpu.


[2026-04-15 10:57:28] [INFO] [model_loader] GPU Memory: 86.2GB allocated, 86.2GB reserved
[2026-04-15 10:57:28] [INFO] [model_loader] Successfully loaded QwQ-32B
[2026-04-15 11:04:15] [INFO] [inference] Generated 20/200
[2026-04-15 11:11:01] [INFO] [inference] Generated 40/200
[2026-04-15 11:17:47] [INFO] [inference] Generated 60/200
[2026-04-15 11:24:32] [INFO] [inference] Generated 80/200
[2026-04-15 11:31:18] [INFO] [inference] Generated 100/200
[2026-04-15 11:38:04] [INFO] [inference] Generated 120/200
[2026-04-15 11:44:49] [INFO] [inference] Generated 140/200
[2026-04-15 11:51:35] [INFO] [inference] Generated 160/200
[2026-04-15 11:58:21] [INFO] [inference] Generated 180/200
[2026-04-15 12:05:07] [INFO] [inference] Generated 200/200

--- DEBUG ---
OUTPUT: Okay, let's see. Janet's ducks lay 16 eggs each day. She uses three eggs for breakfast every morning. Then she bakes muffins using four eggs for her friends. The rest she sells at the market for $2 pe
PRED: 18
GOLD: 18

--- DEBUG

## 5. Faithfulness Profiling (Experiment 2)

Compute SIG, CNS, and RFI metrics for all models.

In [7]:
from scripts.experiments.exp_faithfulness_profiling import run_faithfulness_profiling

faithfulness_results = run_faithfulness_profiling()

print('\n=== Faithfulness Profiling Complete ===')
for model_key, benchmarks in faithfulness_results.items():
    print(f'\n{MODEL_REGISTRY[model_key].short_name}:')
    for bench_key, result in benchmarks.items():
        rfi = result.get('rfi_aggregate', {}).get('mean_rfi', 0)
        print(f'  {bench_key}: RFI = {rfi:.4f}')

[2026-04-15 21:35:39] [INFO] [exp_faithfulness] === Faithfulness profiling: DS-R1-Qwen-7B ===
[2026-04-15 21:35:39] [INFO] [model_loader] Loading model DS-R1-Qwen-7B (deepseek-ai/DeepSeek-R1-Distill-Qwen-7B) in bfloat16...


`torch_dtype` is deprecated! Use `dtype` instead!


[2026-04-15 21:35:41] [ERROR] [exp_faithfulness] Failed to load DS-R1-Qwen-7B: FlashAttention2 has been toggled on, but it cannot be used due to the following error: the package for FlashAttention2 doesn't seem to be installed.
[2026-04-15 21:35:41] [INFO] [exp_faithfulness] === Faithfulness profiling: DS-R1-Llama-8B ===
[2026-04-15 21:35:41] [INFO] [model_loader] Loading model DS-R1-Llama-8B (deepseek-ai/DeepSeek-R1-Distill-Llama-8B) in bfloat16...


config.json:   0%|          | 0.00/826 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Cancellation requested; stopping current tasks.


KeyboardInterrupt: 

## 6. Perturbation Tests (Experiment 3)

In [ ]:
from scripts.experiments.exp_perturbation_tests import run_perturbation_tests

perturbation_results = run_perturbation_tests()
print('\n=== Perturbation Tests Complete ===')

## 7. Failure Taxonomy (Experiment 4)

In [ ]:
from scripts.experiments.exp_failure_classification import run_failure_classification

failure_results = run_failure_classification()
print('\n=== Failure Classification Complete ===')

## 8. Ablation Studies

Run all 5 ablation experiments.

In [ ]:
# Ablation 1: Temperature
from scripts.ablations.ablation_temperature import run_temperature_ablation
temp_results = run_temperature_ablation()
print('Temperature ablation complete.')

In [ ]:
# Ablation 2: CoT Length
from scripts.ablations.ablation_cot_length import run_cot_length_ablation
length_results = run_cot_length_ablation()
print('CoT length ablation complete.')

In [ ]:
# Ablation 3: Perturbation Type Comparison
from scripts.ablations.ablation_perturbation_type import run_perturbation_type_ablation
pert_type_results = run_perturbation_type_ablation()
print('Perturbation type ablation complete.')

In [ ]:
# Ablation 4: Prompt Format
from scripts.ablations.ablation_prompt_format import run_prompt_format_ablation
format_results = run_prompt_format_ablation()
print('Prompt format ablation complete.')

In [ ]:
# Ablation 5: Model Scaling
from scripts.ablations.ablation_model_scaling import run_model_scaling_analysis
scaling_results = run_model_scaling_analysis()
print('Model scaling ablation complete.')

## 9. Cross-Model Analysis (Experiment 5)

In [ ]:
from scripts.experiments.exp_cross_model_analysis import run_cross_model_analysis

cross_model_results = run_cross_model_analysis()

# Print key findings
inverse = cross_model_results.get('inverse_scaling_test', {})
print(f'\nInverse Scaling Test:')
print(f'  Correlation (params vs RFI): {inverse.get("correlation", "N/A")}')
print(f'  Detected: {inverse.get("inverse_scaling_detected", "N/A")}')
print(f'  Interpretation: {inverse.get("interpretation", "N/A")}')

## 10. Results & Tables

In [ ]:
from scripts.visualization.generate_tables import generate_all_tables

tables = generate_all_tables()
print(tables)

## 11. Visualization

In [ ]:
from scripts.visualization.plot_heatmaps import (
    plot_accuracy_heatmap, plot_faithfulness_heatmap, plot_perturbation_heatmap
)
from scripts.visualization.plot_radar_charts import plot_failure_radar, plot_step_type_radar
from scripts.visualization.plot_scaling_curves import plot_scaling_curves, plot_accuracy_faithfulness_scatter
from scripts.visualization.plot_step_information import plot_aggregate_step_info

# Generate all visualizations
plot_accuracy_heatmap()
plot_faithfulness_heatmap()
plot_perturbation_heatmap()
plot_failure_radar()
plot_step_type_radar()
plot_scaling_curves()
plot_accuracy_faithfulness_scatter()
plot_aggregate_step_info()

print(f'\nAll figures saved to: {PATHS["figures_dir"]}')

In [ ]:
# Display generated figures
from IPython.display import Image, display
import glob

fig_dir = PATHS['figures_dir']
for fig_path in sorted(glob.glob(os.path.join(fig_dir, '*.png'))):
    print(f'\n--- {os.path.basename(fig_path)} ---')
    display(Image(filename=fig_path, width=800))

## 12. Sampling & Inspection

Sample individual CoT examples to qualitatively inspect reasoning patterns.

In [ ]:
# Sample from baseline predictions
import random

def sample_and_display(model_key, bench_key, n=3, filter_correct=None):
    """Sample and display CoT examples."""
    pred_path = os.path.join(
        PATHS['raw_results_dir'], 'baseline', f'predictions_{model_key}.json'
    )
    if not os.path.exists(pred_path):
        print(f'No predictions found for {model_key}')
        return
    
    with open(pred_path) as f:
        pred_data = json.load(f)
    
    predictions = pred_data.get('predictions', {}).get(bench_key, [])
    
    if filter_correct is not None:
        predictions = [p for p in predictions if p['is_correct'] == filter_correct]
    
    if not predictions:
        print('No matching examples found.')
        return
    
    samples = random.sample(predictions, min(n, len(predictions)))
    
    model_name = MODEL_REGISTRY[model_key].short_name
    print(f'\n{"="*60}')
    print(f'Model: {model_name} | Benchmark: {bench_key}')
    print(f'Filter: {"correct" if filter_correct else "incorrect" if filter_correct is False else "all"}')
    print(f'{"="*60}')
    
    for i, sample in enumerate(samples):
        print(f'\n--- Sample {i+1} ---')
        print(f'Gold: {sample["gold_answer"]}')
        print(f'Predicted: {sample["predicted_answer"]}')
        print(f'Correct: {sample["is_correct"]}')
        print(f'Steps: {sample["num_steps"]}, Tokens: {sample["num_tokens"]}')
        print(f'\nReasoning:\n{sample["raw_output"]}')
        print()

# Sample correct examples
for model_key in list(MODEL_REGISTRY.keys())[:2]:  # First 2 models
    sample_and_display(model_key, 'gsm8k', n=2, filter_correct=True)
    sample_and_display(model_key, 'gsm8k', n=2, filter_correct=False)

In [ ]:
# Analyze CoT structure of sampled examples
from src.utils.cot_parser import CoTParser

parser = CoTParser()

def analyze_cot_structure(raw_output):
    """Analyze and display CoT structure."""
    parsed = parser.parse(raw_output)
    print(f'Total Steps: {parsed.num_steps}')
    print(f'Has Think Tags: {parsed.has_think_tags}')
    print(f'Final Answer: {parsed.final_answer_text}')
    print('\nStep Breakdown:')
    for step in parsed.steps:
        print(f'  [{step.step_type.upper():15s}] {step.text[:80]}...')
    return parsed

# Analyze a sample
pred_path = os.path.join(PATHS['raw_results_dir'], 'baseline', 'predictions_ds-r1-qwen-7b.json')
if os.path.exists(pred_path):
    with open(pred_path) as f:
        preds = json.load(f)
    gsm8k_preds = preds.get('predictions', {}).get('gsm8k', [])
    if gsm8k_preds:
        print('=== CoT Structure Analysis ===')
        analyze_cot_structure(gsm8k_preds[0]['raw_output'])

## Summary

This notebook runs the complete **FaithCoT** pipeline:
- **5 Benchmarks**: GSM8K, MATH, StrategyQA, ARC-Challenge, FOLIO
- **5 Models**: DS-R1-Qwen-7B, DS-R1-Llama-8B, DS-R1-Qwen-14B, QwQ-32B, DS-R1-Qwen-32B
- **3 Novel Metrics**: SIG, CNS, RFI
- **6-Category Failure Taxonomy**
- **5 Perturbation Tests**: Early answering, mistake injection, step shuffling, step deletion, paraphrasing
- **5 Ablation Studies**: Temperature, CoT length, perturbation type, prompt format, model scaling
- **4+ Result Tables** and **8+ Visualizations**